# Official plots
Each cell below regenerates one figure from the CSV data under `tau_summary_*` symlinks.

In [ ]:
# Shared style + imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from matplotlib.lines import Line2D

sns.set_context("paper")
sns.set_style(
    "whitegrid",
    rc={
        "axes.edgecolor": "black",
        "axes.linewidth": 2.5,
        "axes.spines.top": True,
        "axes.spines.right": True,
        "grid.color": "0.85",
        "grid.linewidth": 0.8,
        "grid.alpha": 1.0,
    },
)
sns.set_palette("colorblind")

# Pull palette colors so explicit color= args use them.
# Colorblind palette: 0=blue, 1=orange, 2=green, 3=red(vermilion), 4=purple,
#                     5=brown, 6=pink, 7=gray, 8=yellow, 9=sky.
CB = sns.color_palette()
C_ORIG, C_RECON, C_FAITH = CB[3], CB[2], CB[0]

CUB = Path("tau_summary_cub50_target_only_official")
PAD = Path("tau_summary_l15_signmode")

## CUB-50: accuracy vs τ (top-5)

In [ ]:
df = pd.read_csv(CUB / "acc_vs_tau_top5.csv")

fig, ax = plt.subplots(figsize=(6, 6))
metrics = [
    ("acc_orig",  r"acc_origin($\tilde{y}_L,\hat{y}$)",                  C_ORIG),
    ("acc_recon", r"acc_recon($\tilde{y}_L,\tilde{y}_{\mathrm{AE}}$)",   C_RECON),
    ("faith",     r"faith($\tilde{y}_{\mathrm{AE}},\hat{y}$)",           C_FAITH),
]
for key, label, color in metrics:
    ax.errorbar(df.tau, df[f"DPO_{key}_top5_mean"], yerr=df[f"DPO_{key}_top5_sem"],
                marker="o", capsize=3, lw=1.5, color=color, ecolor=color, elinewidth=1)
    ax.plot(df.tau, df[f"lit_{key}_top5"], marker="s", lw=1.2, linestyle="--", color=color)

color_handles = [Line2D([0],[0], color=c, lw=2, label=lbl) for _, lbl, c in metrics]
style_handles = [
    Line2D([0],[0], color="black", marker="o", lw=1.5, label="DPO"),
    Line2D([0],[0], color="black", marker="s", lw=1.2, linestyle="--", label="literal"),
]
ax.legend(handles=color_handles + style_handles, loc="best", fontsize=11, frameon=True)
ax.set_xlabel(r"$\tau$", fontsize=18)
ax.set_ylabel("top-5 accuracy", fontsize=18)
ax.tick_params(axis="both", labelsize=14)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(CUB / "acc_vs_tau_top5.pdf")
plt.show()

## CUB-50: DPO − literal (top-5)

In [ ]:
df = pd.read_csv(CUB / "acc_vs_tau_top5.diff.csv")

fig, ax = plt.subplots(figsize=(16/2, 9/2))
metrics = [
    ("acc_orig",  r"acc_origin($\tilde{y}_L,\hat{y}$)",                  C_ORIG),
    ("acc_recon", r"acc_recon($\tilde{y}_L,\tilde{y}_{\mathrm{AE}}$)",   C_RECON),
    ("faith",     r"faith($\tilde{y}_{\mathrm{AE}},\hat{y}$)",           C_FAITH),
]
for key, label, color in metrics:
    ax.errorbar(df.tau, df[f"diff_{key}_top5_mean"], yerr=df[f"diff_{key}_top5_sem"],
                marker="o", capsize=3, lw=1.5, color=color, ecolor=color, elinewidth=1, label=label)
ax.axhline(0, color=CB[7], linestyle="--")
ax.set_xticks([0, 5, 10, 15, 20], labels=["0", "5", "10", "15", "20"])
ax.set_yticks([-0.1, 0, 0.1, 0.2], labels=["-0.1", "0.0", "0.1", "0.2"])
# YLIM = (-0.15, 0.35)
# ax.set_ylim(*YLIM)

ax.set_xlabel(r"$\tau$", fontsize=18)
ax.set_ylabel("pragmatic − literal", fontsize=18)
ax.tick_params(axis="both", labelsize=18)
ax.grid(True)
# ax.legend(loc="best", fontsize=18, frameon=True)
fig.tight_layout()
fig.savefig(CUB / "acc_vs_tau_top5_diff.pdf", bbox_inches="tight")
plt.show()

## CUB-50: concept CDF AUC vs τ

In [ ]:
df = pd.read_csv(CUB / "cdf_auc.csv")
pi_auc = float(df["pi_AUC"].iloc[0])

# Gap from the prior's CDF AUC (lower = closer to the prior).
dpo_gap_mean = pi_auc - df.DPO_AUC_mean
dpo_gap_sem  = df.DPO_AUC_sem
lit_gap      = pi_auc - df.lit_AUC

fig, ax = plt.subplots(figsize=(16/2, 9/2))
ax.errorbar(df.tau, dpo_gap_mean, yerr=dpo_gap_sem,
            marker="o", capsize=3, lw=1.5, color=CB[3],
            ecolor=CB[3], elinewidth=1, label="pragmatic")
ax.plot(df.tau, lit_gap, lw=1.5, linestyle="-",
        color="black", label="literal")
# ax.axhline(0, color="black", lw=0.6, alpha=0.5)
ax.set_xticks([0, 5, 10, 15, 20], labels=["0", "5", "10", "15", "20"])
ax.set_xlabel(r"$\tau$", fontsize=18)
ax.set_ylabel(r"AUC$(\pi_k)$ $-$ AUC(speaker)", fontsize=18)
ax.tick_params(axis="both", labelsize=18)
ax.grid(True)
ax.legend(loc="best", fontsize=14, frameon=True)
fig.tight_layout()
fig.savefig(CUB / "cdf_auc.pdf")
plt.show()

## CUB-50: attribute group distribution pies
Each pie = one CSV in `cub_prior/`. Slice colors are fixed across pies so groups are visually comparable. Labels shown on slices ≥ 3%.

In [ ]:
PRIOR_DIR = Path("cub_prior")

# Fix the group order using the MTurk CSV (column order is consistent across files)
ref_df = pd.read_csv(PRIOR_DIR / "prior_mturk.csv")
GROUPS = list(ref_df.columns)
GROUP_LABELS = [g.replace("has_", "") for g in GROUPS]

# Global color map: one fixed color per group (combine palettes to get 28 distinct hues)
group_colors = (
    sns.color_palette("tab20", 20)
    + sns.color_palette("Set3", 12)[:8]
)
group_colors = group_colors[:len(GROUPS)]


def plot_group_pies(speaker_subdir, suptitle, savename):
    """Plot a 2x4 grid of attribute-group pies.

    Priors come from PRIOR_DIR/; speaker CSVs come from PRIOR_DIR/speaker_subdir/.
    """
    sd = PRIOR_DIR / speaker_subdir
    files = [
        (PRIOR_DIR / "prior_mturk.csv",  "MTurk prior"),
        (PRIOR_DIR / "prior_text.csv",   "Text prior (LLM-extracted)"),
        (sd / "literal_speaker.csv",     "literal speaker"),
        (sd / "dpo_tau0.csv",            r"DPO $\tau=0$"),
        (sd / "dpo_tau5.csv",            r"DPO $\tau=5$"),
        (sd / "dpo_tau10.csv",           r"DPO $\tau=10$"),
        (sd / "dpo_tau15.csv",           r"DPO $\tau=15$"),
        (sd / "dpo_tau20.csv",           r"DPO $\tau=20$"),
    ]

    fig, axes = plt.subplots(2, 4, figsize=(16, 9))
    axes = axes.flatten()
    for ax, (path, title) in zip(axes, files):
        df = pd.read_csv(path)
        values = df.iloc[0].values.astype(float)
        wedges, _, autotexts = ax.pie(
            values,
            labels=None,
            colors=group_colors,
            autopct=lambda p: f"{p:.0f}%" if p >= 3 else "",
            startangle=90,
            wedgeprops={"linewidth": 0.8, "edgecolor": "white"},
            pctdistance=0.75,
        )
        for at in autotexts:
            at.set_fontsize(8)
            at.set_color("white")
        ax.set_title(title, fontsize=12)

    fig.legend(
        handles=[plt.Rectangle((0,0), 1, 1, fc=group_colors[i]) for i in range(len(GROUPS))],
        labels=GROUP_LABELS,
        loc="center right",
        bbox_to_anchor=(1.0, 0.5),
        fontsize=8,
        ncol=1,
        frameon=True,
    )
    fig.subplots_adjust(right=0.85)
    fig.suptitle(suptitle, fontsize=14, y=0.99)
    fig.savefig(PRIOR_DIR / savename, bbox_inches="tight")
    plt.show()


plot_group_pies(
    "final_iter",
    "CUB-50 attribute group distribution per source / speaker (final iter)",
    "group_pies_final_iter.pdf",
)

## CUB-50: attribute group distribution pies (best iter)
Same as above but speaker CSVs come from `cub_prior/best_iter/`, where each DPO seed contributes the iter that maximizes `val/listener_acc_recon` (literal uses its own best iter too).

In [ ]:
plot_group_pies(
    "best_iter",
    "CUB-50 attribute group distribution per source / speaker (best iter)",
    "group_pies_best_iter.pdf",
)

## CUB-50: text prior vs literal vs DPO τ=15 (top-10 groups + "Other")
Top-10 groups by text-prior share, plotted on the original denominator (no renormalization). The remaining 18 groups are pooled into a single "Other" wedge. DPO τ=15 concentrates more mass on the prior's top groups than the literal speaker, so its "Other" wedge is smaller and its internal proportions are closer to the text prior. Uses `final_iter/` CSVs.

In [ ]:
SD = PRIOR_DIR / "final_iter"

# Use text prior to pick & order the top-K groups.
txt_df = pd.read_csv(PRIOR_DIR / "prior_text.csv")
GROUPS_28 = list(txt_df.columns)
txt_vals = txt_df.iloc[0].values.astype(float)
TOP_K = 20
top_idx = np.argsort(-txt_vals)[:TOP_K]
top_groups = [GROUPS_28[i] for i in top_idx]
slice_labels = [g.replace("has_", "") for g in top_groups] + ["Other"]

# husl: TOP_K evenly-spaced hues at uniform saturation/lightness -> a clean,
# muted rainbow with as many distinct colors as needed. Gray = "Other".
top_colors = sns.color_palette("husl", TOP_K)
slice_colors = list(top_colors) + [(0.78, 0.78, 0.78)]

panels = [
    (PRIOR_DIR / "prior_text.csv",   "Human prior"),
    (SD / "literal_speaker.csv",     "Literal speaker"),
    (SD / "dpo_tau20.csv",           r"Pragmatic speaker"),
]

def slice_values(values):
    top = [values[i] for i in top_idx]
    other = float(values.sum() - sum(top))
    return top + [other]

fig, axes = plt.subplots(1, 3, figsize=(15, 5.6))
for ax, (path, title) in zip(axes, panels):
    df = pd.read_csv(path)
    values = df.iloc[0].values.astype(float)
    sl = slice_values(values)
    total = sum(sl)
    wedges, _, autotexts = ax.pie(
        sl,
        labels=None,
        colors=slice_colors,
        autopct=lambda p: f"{p:.0f}%" if p >= 3 else "",
        startangle=90,
        wedgeprops={"linewidth": 0.8, "edgecolor": "white"},
        pctdistance=0.75,
    )
    for at in autotexts:
        at.set_fontsize(9)
        at.set_color("white")
    ax.set_title(title, fontsize=14)

# Shared legend
fig.legend(
    handles=[plt.Rectangle((0, 0), 1, 1, fc=slice_colors[i]) for i in range(len(slice_labels))],
    labels=slice_labels,
    loc="center right",
    bbox_to_anchor=(1.02, 0.5),
    fontsize=10,
    ncol=1,
    frameon=True,
)
fig.subplots_adjust(right=0.83)
# fig.suptitle(f"CUB-50: top-{TOP_K} text-prior groups + Other", fontsize=15, y=1.0)
fig.savefig(PRIOR_DIR / f"pies_text_vs_lit_vs_dpo20_top{TOP_K}.pdf", bbox_inches="tight")
plt.show()

## PadChest: accuracy vs τ

In [ ]:
df = pd.read_csv(PAD / "acc_vs_tau.csv")
dpo = df[df.cond == "DPO_lc0.1"].sort_values("tau").reset_index(drop=True)
lit = df[df.cond == "literal"].sort_values("tau").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(6, 6))
metrics = [
    ("acc_orig",    r"acc_origin($\tilde{y}_L,\hat{y}$)",                  C_ORIG),
    ("acc_recon",   r"acc_recon($\tilde{y}_L,\tilde{y}_{\mathrm{AE}}$)",   C_RECON),
    ("faith_agree", r"faith($\tilde{y}_{\mathrm{AE}},\hat{y}$)",           C_FAITH),
]
for key, label, color in metrics:
    ax.errorbar(dpo.tau, dpo[f"{key}_mean"], yerr=dpo[f"{key}_sem"],
                marker="o", capsize=3, lw=1.5, color=color, ecolor=color, elinewidth=1)
    ax.errorbar(lit.tau, lit[f"{key}_mean"], yerr=lit[f"{key}_sem"],
                marker="s", capsize=3, lw=1.2, linestyle="--", color=color, ecolor=color, elinewidth=1)

color_handles = [Line2D([0],[0], color=c, lw=2, label=lbl) for _, lbl, c in metrics]
style_handles = [
    Line2D([0],[0], color="black", marker="o", lw=1.5, label="DPO"),
    Line2D([0],[0], color="black", marker="s", lw=1.2, linestyle="--", label="literal"),
]
ax.legend(handles=color_handles + style_handles, loc="best", fontsize=11, frameon=True)
ax.set_xlabel(r"$\tau$", fontsize=18)
ax.set_ylabel("accuracy", fontsize=18)
ax.tick_params(axis="both", labelsize=14)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(PAD / "acc_vs_tau.pdf")
plt.show()

## PadChest: pragmatic − literal

In [ ]:
df = pd.read_csv(PAD / "acc_vs_tau.diff.csv")

fig, ax = plt.subplots(figsize=(16/2, 9/2))
metrics = [
    ("acc_orig",    r"agree_origin($\tilde{y}_L,\hat{y}$)",                  C_ORIG),
    ("acc_recon",   r"agree_recon($\tilde{y}_L,\tilde{y}_{\mathrm{AE}}$)",   C_RECON),
    ("faith_agree", r"agree_faith($\tilde{y}_{\mathrm{AE}},\hat{y}$)",           C_FAITH),
]
for key, label, color in metrics:
    ax.errorbar(df.tau, df[f"diff_{key}_mean"], yerr=df[f"diff_{key}_sem"],
                marker="o", capsize=3, lw=1.5, color=color, ecolor=color, elinewidth=1, label=label)
ax.axhline(0, color=CB[7], linestyle="--")
# ax.set_yticks([-0.1, 0, 0.1, 0.2, 0.3], labels=["-0.1", "0.0", "0.1", "0.2", "0.3"])
YLIM = (-0.15, 0.35)
ax.set_ylim(*YLIM)

ax.set_xlabel(r"$\tau$", fontsize=18)
ax.set_ylabel("pragmatic − literal", fontsize=18)
ax.tick_params(axis="both", labelsize=18)
ax.grid(True)
ax.legend(loc="best", fontsize=18, frameon=True)
fig.tight_layout()
fig.savefig(PAD / "acc_vs_tau_diff.pdf")
plt.show()

## PadChest: per-class concept CDFs (τ=6 and τ=10)

In [ ]:
classes = ["copd_signs", "pneumonia", "heart_insufficiency"]
title_name = ["copd", "pneumonia", "heart insufficiency"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for ax, cname, tname in zip(axes, classes, title_name):
    df = pd.read_csv(PAD / f"concept_cdf_{cname}_tau6_tau10.csv")
    ax.plot(df["rank"], df["pi_cdf"], color=CB[7], ls="--", lw=1.4, label=r"$\pi_k$ (radiologist)")
    ax.plot(df["rank"], df["literal_cdf"], color="black", lw=1.6, label="literal")
    # DPO τ=6 with SEM band
    m6, s6 = df["dpo_tau6_cdf_mean"], df["dpo_tau6_cdf_sem"]
    ax.plot(df["rank"], m6, color=CB[1], lw=1.6, label=r"DPO $\tau=6$")
    ax.fill_between(df["rank"], m6 - s6, m6 + s6, color=CB[1], alpha=0.25, linewidth=0)
    # DPO τ=10 with SEM band
    m10, s10 = df["dpo_tau10_cdf_mean"], df["dpo_tau10_cdf_sem"]
    ax.plot(df["rank"], m10, color=CB[3], lw=1.6, label=r"DPO $\tau=10$")
    ax.fill_between(df["rank"], m10 - s10, m10 + s10, color=CB[3], alpha=0.25, linewidth=0)
    ax.set_title(tname, fontsize=16)
    ax.set_xlabel(r"concept rank by $\pi_k$", fontsize=16)
    ax.tick_params(axis="both", labelsize=16)
    ax.grid(True, alpha=0.5)

axes[0].set_ylabel("cumulative selection mass", fontsize=16)
axes[0].legend(loc="lower right", fontsize=14, frameon=True)
fig.tight_layout()
fig.savefig(PAD / "concept_cdfs_tau6_tau10.pdf")
plt.show()

# Appendix
The $\lambda_c$ and $\gamma_{\mathrm{faith}}$ sweeps of the appendix are not part of this release; only the base-classifier training curves are reproduced here.


## Appendix: base classifier training curves
PadChest ViT validation macro balanced-accuracy vs. step, and CUB MLP-on-CLIP validation top-1 accuracy vs. step. Single-seed runs exported from W&B; the `__MIN`/`__MAX` columns are duplicates of the value (no smoothing band to draw).

In [ ]:
APX = Path("appendix")
BASE_CLS = APX / "base_classifier"
BASE_CLS.mkdir(parents=True, exist_ok=True)

def load_curve(path):
    """Wandb-exported CSV: first col Step, second col the metric value."""
    df = pd.read_csv(path)
    return df.iloc[:, 0].astype(int).to_numpy(), df.iloc[:, 1].astype(float).to_numpy()

def plot_base_cls(step, val, ylabel, color, savepath):
    fig, ax = plt.subplots(figsize=(16/2, 9/2))
    ax.semilogx(step, val, lw=1.8, color=color)
    ax.set_xlabel("Training samples", fontsize=18)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.tick_params(axis="both", labelsize=18)
    ax.grid(True)
    fig.tight_layout()
    fig.savefig(savepath)
    plt.show()

step_pad, val_pad = load_curve(BASE_CLS / "base_classifier_pad.csv")
step_cub, val_cub = load_curve(BASE_CLS / "base_classifier_cub.csv")

plot_base_cls(step_pad, val_pad,
              "Accuracy", CB[4],
              BASE_CLS / "pad_base_classifier.pdf")
plot_base_cls(step_cub, val_cub,
              "Accuracy", CB[4],
              BASE_CLS / "cub_base_classifier.pdf")